In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# Research Agent with MCP

This is just an added example to showcase MCP features by accessing a file in local system with controlled access and granular permission management.

**What It Provides:**
- **File operations** with strict access control
- **Directory management** with dynamic permissions
- **Search capabilities** across allowed directories
- **Metadata access** for files and directories

**Available Tools:**
- **File Operations**: `read_file`, `write_file`, `edit_file`, `read_multiple_files`
- **Directory Management**: `create_directory`, `list_directory`, `move_file`
- **Search & Discovery**: `search_files`, `get_file_info`, `list_allowed_directories`

### Design choices

**Why MCP setup needs `async` methods?**
- Server communications (connection setup, logging, notifications, etc.)
- Tool invocations could be slow.
- I/O operations are naturally async, and one blocking sync call could freeze whole communication.
- Async enables non-blocking, concurrent operations.
- In langchain, all MCP adapters use async-only operations.


### Simple MCP Example

In [3]:
# ! pip install -q langchain-mcp-adapters

In [ ]:
# import os
# from langchain_mcp_adapters.client import MultiServerMCPClient
# from rich.console import Console
# from rich.panel import Panel
# from rich.table import Table

# console = Console()

# # Fetch absolute path to the example file
# doc_path = os.path.abspath(".assets/")

# # Check if directory exists
# if os.path.exists(doc_path):
#     console.print(f"[green]✓ Directory exists with files:[/green] {os.listdir(doc_path)}")
# else:
#     console.print("[red]✗ Directory does not exist![/red]")

# # MCP Client setup
# import os, sys

# mcp_config = {
#     "filesystem": {
#         "command": "npx",
#         "args": [
#             "-y",
#             "@modelcontextprotocol/server-filesystem",
#             doc_path,
#         ],
#         "transport": "stdio",
#     }
# }


# client = MultiServerMCPClient(mcp_config)

# # Check available Tools
# tools = await client.get_tools()
# table = Table(title="Available MCP Tools", show_header=True, header_style="bold magenta")
# table.add_column("Tool Name", style="cyan", width=25)
# table.add_column("Description", style="white", width=80)

# for tool in tools:
#     description = tool.description[:77] + "..." if len(tool.description) > 80 else tool.description
#     table.add_row(tool.name, description)
# console.print(table)

## Agent with MCP Setup

### States & Schema

In [2]:
import operator
from typing import TypedDict, Annotated, List, Sequence
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

########## STATE DEFINITIONS ##########

class ResearcherState(TypedDict):
    """
    State for the research agent containing message history and research metadata.
    
    This state tracks the researcher's conversation, iteration count for limiting
    tool calls, the research topic being investigated, compressed findings,
    and raw research notes for detailed analysis.
    """
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages] # A sequence of messages representing the conversation history of the researcher.
    tool_call_iterations: int
    research_topic: str
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]

class ResearcherOutputState(TypedDict):
    """
    Output state for the research agent containing final research results.
    
    This represents the final output of the research process with compressed
    research findings and all raw notes from the research process.
    """
    compressed_research: str
    raw_notes: Annotated[List[str], operator.add]
    researcher_messages: Annotated[Sequence[BaseMessage], add_messages]

########## OUTPUT SCHEMAS ##########

class ClarifyUserRequest(BaseModel): # From Scope Agent
    """Structured output for requesting clarification from the user."""
    need_clarification: Annotated[bool, Field(..., description="Indicates whether clarification is needed from the user.")] 
    question_to_user: Annotated[str, Field(default="", description="The question to ask the user for clarification. Required only when need_clarification=True; otherwise empty string.")]
    verification: Annotated[str, Field(default="", description="Acknowledgment message confirming the user's request is sufficient to proceed with research. Required only when need_clarification=False; otherwise empty string.")]

class ResearchBrief(BaseModel): # From Scope Agent
    """Structured output for the research brief generated from user input."""
    research_brief: Annotated[str, Field(..., description="A concise research brief summarizing the user's request and the research objectives to guide the research agents.")]

class WebSearchSummary(BaseModel):
    """Schema for webpage content summarization."""
    summary: str = Field(description="Concise summary of the webpage content")
    key_points: str = Field(description="Important quotes and excerpts from the content")

### Core setup

In [ ]:
# %%writefile ./src/research_agent_mcp.py

"""
Research Agent with MCP Integration.

This module implements a research agent that integrates with Model Context Protocol (MCP)
servers to access tools and resources. The agent demonstrates how to use MCP filesystem
server for local document research and analysis.

Key features:
- MCP server integration for tool access
- Async operations for concurrent tool execution (required by MCP protocol)
- Filesystem operations for local document research
- Secure directory access with permission checking
- Research compression for efficient processing
- Lazy MCP client initialization for LangGraph Platform compatibility
- Cached MCP tool list to avoid per-turn round trips
- Hard iteration cap to prevent runaway tool loops
"""

from typing import Literal
from IPython.display import Image, display
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, filter_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END

# Custom imports
from src.agent_prompts import research_agent_prompt_with_mcp, compress_research_system_prompt, compress_research_human_message
from src.utils import date_today_str, current_dir
from src.agent_tools import think_tool

# ===== CONFIGURATION =====

doc_path = str(current_dir() / "assets")

# MCP server configuration for filesystem access
mcp_config = {
    "filesystem": {
        "command": "npx",
        "args": [
            "-y",  # Auto-install if needed
            "@modelcontextprotocol/server-filesystem",
            doc_path 
        ],
        "transport": "stdio"  # Communication via stdin/stdout
    }
}

# Global client + tool cache - both initialized lazily.
_client = None
_mcp_tools_cache = None

def get_mcp_client():
    """Get or initialize MCP client lazily to avoid issues with LangGraph Platform."""
    global _client
    if _client is None:
        _client = MultiServerMCPClient(mcp_config)
    return _client

async def _get_mcp_tools():
    """Fetch MCP tool list once and cache it. Tool list is static for the session."""
    global _mcp_tools_cache
    if _mcp_tools_cache is None:
        client = get_mcp_client()
        _mcp_tools_cache = await client.get_tools()
    return _mcp_tools_cache

# Initialize models with deterministic temperature for reproducibility.
compress_model = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=32000, temperature=0)
model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

MAX_TOOL_ITERATIONS = 5

# ===== AGENT NODES =====

async def llm_call(state: ResearcherState):
    """Analyze current state and decide on tool usage with MCP integration.

    This node:
    1. Retrieves available tools from MCP server (cached after first fetch)
    2. Binds tools to the language model
    3. Processes user input asynchronously (await ainvoke - never block the event loop)

    Returns updated state with model response.
    """
    mcp_tools = await _get_mcp_tools()
    tools = mcp_tools + [think_tool]

    model_with_tools = model.bind_tools(tools)

    researcher_system_message = research_agent_prompt_with_mcp.format(date=date_today_str())

    response = await model_with_tools.ainvoke(
        [SystemMessage(content=researcher_system_message)] + state["researcher_messages"]
    )

    return {"researcher_messages": [response]}

async def tool_node(state: ResearcherState):
    """Execute tool calls using MCP tools.

    MCP tools require ainvoke (async IPC with the MCP server subprocess).
    think_tool is sync, so we keep regular invoke for that one.
    """
    tool_calls = state["researcher_messages"][-1].tool_calls

    mcp_tools = await _get_mcp_tools()
    tools = mcp_tools + [think_tool]
    tools_by_name = {tool.name: tool for tool in tools}

    observations = []
    for tool_call in tool_calls:
        tool = tools_by_name[tool_call["name"]]
        if tool_call["name"] == "think_tool":
            observation = tool.invoke(tool_call["args"])
        else:
            observation = await tool.ainvoke(tool_call["args"])
        observations.append(observation)

    tool_outputs = [
        ToolMessage(
            content=observation,
            name=tool_call["name"],
            tool_call_id=tool_call["id"],
        )
        for observation, tool_call in zip(observations, tool_calls)
    ]

    return {
        "researcher_messages": tool_outputs,
        "tool_call_iterations": state.get("tool_call_iterations", 0) + 1,
    }

def compress_research(state: ResearcherState) -> dict:
    """Compress research findings into a concise summary.

    Takes all the research messages and tool outputs and creates
    a compressed summary suitable for further processing or reporting.

    This function filters out think_tool calls and focuses on substantive
    file-based research content from MCP tools.
    """

    compress_research_system_message = compress_research_system_prompt.format(date=date_today_str())
    compress_research_human = compress_research_human_message.format(
        research_topic=state.get("research_topic", "")
    )
    messages = [
        SystemMessage(content=compress_research_system_message)] + state.get("researcher_messages", []) + [HumanMessage(content=compress_research_human)]

    response = compress_model.invoke(messages)

    # Extract raw notes from tool and AI messages
    raw_notes = [
        str(m.content) for m in filter_messages(
            state["researcher_messages"], 
            include_types=["tool", "ai"]
        )
    ]

    return {
        "compressed_research": str(response.content),
        "raw_notes": ["\n".join(raw_notes)]
    }

# ===== ROUTING LOGIC =====

def should_continue(state: ResearcherState) -> Literal["tool_node", "compress_research"]:
    """Determine whether to continue with tool execution or compress research.

    Stops when the LLM stops calling tools OR the iteration cap is reached.
    """
    last_message = state["researcher_messages"][-1]

    if not last_message.tool_calls:
        return "compress_research"

    if state.get("tool_call_iterations", 0) >= MAX_TOOL_ITERATIONS:
        return "compress_research"

    return "tool_node"

# ===== GRAPH CONSTRUCTION =====

# Build the agent workflow
agent_builder_mcp = StateGraph(ResearcherState, output_schema=ResearcherOutputState)

# Add nodes to the graph
agent_builder_mcp.add_node("llm_call", llm_call)
agent_builder_mcp.add_node("tool_node", tool_node)
agent_builder_mcp.add_node("compress_research", compress_research)

# Add edges to connect nodes
agent_builder_mcp.add_edge(START, "llm_call")
agent_builder_mcp.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "tool_node": "tool_node",        # Continue to tool execution
        "compress_research": "compress_research",  # Compress research findings
    },
)
agent_builder_mcp.add_edge("tool_node", "llm_call")  # Loop back for more processing
agent_builder_mcp.add_edge("compress_research", END)

# Compile the agent
agent_mcp = agent_builder_mcp.compile()
display(Image(agent_mcp.get_graph(xray=True).draw_mermaid_png()))

## Testing

In [ ]:
from src.utils import format_messages

# sample output from scope agent run
research_brief = """
I am learning about Agentic AI best practices for production environments. Please provide a comprehensive,         
educational deep-dive covering all of the following areas as they apply to real-world production deployments of    
agentic AI systems:                                                                                                

 1. System Design & Architecture - Best practices for designing agentic AI systems (e.g., agent orchestration,      
   multi-agent frameworks, tool use, memory management, context window handling, modularity, and scalability).     
 2. Safety, Alignment & Guardrails - How to implement effective guardrails, constrain agent behavior, prevent       
   unintended actions, ensure alignment with intended goals, and handle failure modes gracefully in production.    
 3. Security & Risk Management - Key security considerations such as prompt injection, data leakage, privilege      
   escalation, supply chain risks, and how to mitigate them in agentic pipelines.                                  
 4. Deployment & Operations - Production deployment strategies, monitoring, observability, logging, tracing agent   
   actions, debugging, latency management, cost control, and reliability engineering for agentic systems.          
 5. Governance & Ethics - Responsible AI practices, human-in-the-loop design, auditability, accountability,         
   regulatory considerations, and ethical frameworks relevant to deploying autonomous agents.                      

The research should be grounded in current (up to May 2026) industry knowledge, frameworks, and real-world         
production experience. Prioritize authoritative and primary sources such as official documentation from leading AI 
labs and frameworks (e.g., OpenAI, Anthropic, Google DeepMind, LangChain, AutoGen, CrewAI), peer-reviewed research,
and well-regarded engineering blogs (e.g., from companies like Databricks, Hugging Face, or similar). The audience 
is a learner seeking to understand and apply these best practices in real production settings, so clarity, concrete
examples, and actionable guidance are important.        
"""

response = await agent_mcp.ainvoke( # IMPORTANT: Must use await, async invoke for MCP agent invocation
    {
        "researcher_messages": [HumanMessage(content=research_brief)]
    }
)

format_messages(response["researcher_messages"])